# RHUH-GBM: kiểm tra dữ liệu và xem ảnh kèm nhãn

Notebook này chỉ cấu hình và gọi hàm từ package `rhuh_qc/` trong repo.
Để dùng Colab: **push các file code mới lên Git → mở notebook → clone repo ở cell đầu → mount Drive → sửa `ROOT` → chạy các cell cần dùng**.

Hai nguồn ảnh được hiển thị riêng:

- **NIfTI đã xử lý + mask:** chọn patient/timepoint/modality và kéo lát. Mỗi thời điểm có 5 MRI và một mask chung.
- **DICOM gốc:** xem series từ ZIP hoặc folder đã giải nén. Chưa ghép mask atlas lên DICOM native khi chưa có transform tương ứng.

Chỉ đọc dữ liệu nguồn; các bảng kết quả nằm trong RAM. Dữ liệu trên Drive không đi kèm khi clone repo vì `data/` được Git bỏ qua.

## 1. Clone repo và cài thư viện QC

`REPO_URL` trỏ tới repo hiện tại. Nếu push sang fork khác, sửa URL bên dưới.
Trong Jupyter local, cell tự dùng repo chứa package. Trong Colab mới, cell clone vào `/content/MedSegFactory`.

Nếu Colab đã clone repo từ lần trước, sau khi push code mới hãy đặt `UPDATE_REPO=True` rồi chạy cell clone/update. Tiếp theo chạy lại cell import và mục 4 để tạo viewer mới với callback mới.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/AIVIETNAM-AIO-Tuan/MedSegFactory.git"
UPDATE_REPO = False  # True: git pull code moi vao ban clone dang dung.
REPO_DIR = next(
    (p.resolve() for p in (Path.cwd(), Path.cwd().parent) if (p / "rhuh_qc").is_dir()),
    Path("/content/MedSegFactory"),
)
if not (REPO_DIR / "rhuh_qc").is_dir():
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    if not (REPO_DIR / "rhuh_qc").is_dir():
        raise RuntimeError("Repo chưa có rhuh_qc/. Hãy push code mới, hoặc cập nhật bản clone cũ bằng git pull.")
if UPDATE_REPO:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repository:", REPO_DIR)

In [ ]:
%pip install -q -r requirements-qc.txt

## 2. Mount Drive, cấu hình và import hàm

Sửa `ROOT` thành folder RHUH-GBM của bạn. Folder có thể chứa cả gói NIfTI và DICOM ZIP.
`CHECK_IMAGE_VOXELS=True` đọc toàn bộ MRI để tìm NaN/Inf; có thể mất vài phút trên Drive. `False` chỉ kiểm tra header MRI, nhưng vẫn đọc và kiểm tra đầy đủ mask.

Bản local có MRI thiếu spatial-unit metadata. `ALLOW_UNKNOWN_IMAGE_UNITS=True` cho phép **giả định đơn vị của nhãn khi hiển thị** chỉ khi shape/affine số khớp, đồng thời báo WARN; đặt `False` để từ chối. File nguồn không bị sửa.

In [ ]:
IN_COLAB = False
try:
    from google.colab import drive, output
except ImportError:
    pass
else:
    IN_COLAB = True
    drive.mount("/content/drive")
    output.enable_custom_widget_manager()

ROOT = (Path("/content/drive/MyDrive/RHUH-GBM") if IN_COLAB
        else REPO_DIR / "data" / "RHUH-GBM")
CHECK_IMAGE_VOXELS = True
ALLOW_UNKNOWN_IMAGE_UNITS = True
print("Dataset:", ROOT)

In [ ]:
%matplotlib inline
import importlib
import rhuh_qc

# Reload ca module con de Colab khong giu callback cu sau khi git pull.
for module_name in (
    "rhuh_qc.nifti", "rhuh_qc.dicom", "rhuh_qc.visualization",
    "rhuh_qc.widgets", "rhuh_qc",
):
    importlib.reload(importlib.import_module(module_name))

from IPython.display import display
from rhuh_qc import (
    discover_cases, audit_cases, create_nifti_viewer,
    inventory_dicom_zips, load_dicom_zip, load_dicom_folder,
    audit_dicom_zips, plot_triptych,
)
print("QC package:", rhuh_qc.__file__)

## 3. Audit NIfTI

Kiểm tra thiếu/trùng file, lỗi đọc, shape/affine, đơn vị, NaN/Inf và giá trị nhãn. Nhãn được giữ nguyên numeric ID; thể tích tính từ grid/đơn vị của mask gốc.
Muốn chỉ kiểm tra một bệnh nhân, thay `CASES` bằng dict con như dòng comment dưới đây.

In [ ]:
CASES = discover_cases(ROOT)
# CASES = {key: case for key, case in CASES.items() if key[0] == "RHUH-0001"}
qc, label_stats = audit_cases(
    CASES, check_image_voxels=CHECK_IMAGE_VOXELS,
    allow_unknown_image_units=ALLOW_UNKNOWN_IMAGE_UNITS,
)
print("Cases:", len(qc), "| Status:", qc.status.value_counts().to_dict())
display(qc)
display(label_stats)

## 4. Viewer NIfTI đã xử lý + nhãn

Mặc định T1w (`t1`); `t1ce` là chuỗi riêng có tương phản. Các mặt phẳng dùng grid hiển thị RAS. Nếu ảnh oblique, chỉ bản hiển thị được resample: MRI tuyến tính, mask nearest-neighbor.

Với RHUH-0001: `0` trước mổ, `1` hậu phẫu sớm, `2` theo dõi tái phát. Tên `RHUH-0001_1_t1.nii.gz` ghép với `RHUH-0001_1_segmentations.nii.gz`; không ghép mask từ thời điểm khác.

Viewer cập nhật PNG trực tiếp trong widget. Đổi lựa chọn hoặc **thả thanh trượt** để cập nhật ảnh ngay bên dưới. Dòng trạng thái hiển thị patient/timepoint, lát cắt và opacity hiện tại; lỗi tải/render cũng hiện tại đây.

In [ ]:
viewer = create_nifti_viewer(
    CASES, default_modality="t1",
    allow_unknown_image_units=ALLOW_UNKNOWN_IMAGE_UNITS,
)
display(viewer)

## 5. Kiểm kê DICOM ZIP

Đếm thành viên DICOM từ danh mục ZIP và đọc **một header mẫu mỗi ZIP**. Bảng này chưa kiểm tra mọi pixel hay bảo đảm đủ lát ở đầu/cuối. Không cộng các bản `.dcm` đã giải nén vào số lượng ZIP, tránh đếm hai lần.

In [ ]:
zip_paths, dicom_inventory = inventory_dicom_zips(ROOT)
print("ZIP archives:", len(zip_paths))
display(dicom_inventory)

## 6. Xem DICOM T1w gốc

Ví dụ dùng series `Ax T1 FSE` của RHUH-0001 trong folder `afd75`. Sửa `DICOM_SERIES_FOLDER` để chọn series khác. Hàm đọc ZIP trực tiếp; nếu folder không có ZIP, đọc các `.dcm` đã giải nén.

Các folder T1w đã xác minh của ca này là `afd75`, `9b7cc`, `6db46` theo thứ tự thời gian. Tên folder/UID không mã hóa trực tiếp giai đoạn. Chuỗi DWI có thể lặp vị trí qua nhiều b-value nên loader sẽ từ chối xếp thành một 3D stack mơ hồ.

In [ ]:
DICOM_SERIES_FOLDER = ROOT / (
    "GC_manifest_RHUH-GBM_20260326/RHUH-0001/"
    "The Rio Hortega University Hospital Glioblastoma datas/afd75"
)
DICOM_ZIP = next(DICOM_SERIES_FOLDER.glob("*.zip"), None)
raw_image = (load_dicom_zip(DICOM_ZIP) if DICOM_ZIP is not None
             else load_dicom_folder(DICOM_SERIES_FOLDER))
plot_triptych(raw_image, title="RHUH-0001 | native DICOM T1w | no mask overlay")

## 7. Tùy chọn: kiểm tra mọi ZIP

Đặt `RUN_FULL_DICOM_QC=True` để kiểm tra toàn bộ header và giải mã pixel của mọi ZIP; bước này có thể lâu. Chuỗi multiframe, mixed series, lặp vị trí hoặc hình học không được hỗ trợ sẽ được báo ERROR; không tự kết luận dữ liệu gốc hỏng.

In [ ]:
RUN_FULL_DICOM_QC = False
if RUN_FULL_DICOM_QC:
    full_dicom_qc = audit_dicom_zips(zip_paths)
    display(full_dicom_qc)

## Diễn giải và nguồn

- `OK`: đạt các kiểm tra đã bật; không chứng minh ranh giới segmentation chính xác bằng mắt. `WARN`: có giả định/cảnh báo cần đọc; `ERROR`: xem cột `issues`.
- Shape/affine khớp trong một visit không tự chứng minh registration giải phẫu giữa các visit trước/sau mổ.
- [Bài báo RHUH-GBM](https://arxiv.org/html/2305.00005) và [TCIA](https://wiki.cancerimagingarchive.net/pages/viewpage.action?pageId=145755234) định nghĩa ID 1 = necrosis, 2 = peritumoral signal alteration (phù + u không bắt thuốc), 3 = enhancing tumor. Không có class khoang mổ riêng được tài liệu hóa. README code của tác giả ghi ID 4 cho enhancing tumor; notebook giữ nguyên ID 3 quan sát trong bản local, không tự remap.
- [Pipeline của tác giả](https://github.com/smcch/RHUH-GBM-dataset-MRI-preprocessing) mô tả quy ước `0/1/2` và ảnh đăng ký vào atlas. NIfTI mask này cần đúng transform để đưa về native DICOM; resize hoặc lật trục không thay thế registration đó.
- Source code: `rhuh_qc/nifti.py`, `dicom.py`, `visualization.py`, `widgets.py`. Môi trường QC dùng `requirements-qc.txt`, không cần cài dependency training/CUDA.